# **Laboratorium 4 - ETL w hurtowniach danych**
# Zadanie 2 - Integracja danych z wielu źródeł

Celem zadania jest integracja danych z dwóch źródeł:
- Online_Retail.csv
- Online_Retail_II.xlsx

Wynikiem jest jedna spójna tabela faktów.

## 1. Extract - wczytanie danych

Wczytujemy dane z dwóch różnych źródeł (CSV i Excel).

In [1]:
import pandas as pd

# CSV (problemowy dataset)
df1 = pd.read_csv(
    "Online_Retail.csv",
    encoding="ISO-8859-1",
    on_bad_lines="skip"
)

# Excel
df2 = pd.read_excel("online_retail_II.xlsx")

df1.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


## 2. Analiza danych

Porównujemy:
- nazwy kolumn
- typy danych
- brakujące wartości

### Czy struktura danych jest identyczna?
Nie, struktura danych może się różnić między plikami (np. nazwy kolumn lub typy danych).

### Czy dane można od razu połączyć?
Nie. Przed połączeniem należy ujednolicić strukturę danych oraz oczyścić dane.

In [2]:
print("DF1 columns:", df1.columns)
print("DF2 columns:", df2.columns)

print("\nDF1 types:\n", df1.dtypes)
print("\nDF2 types:\n", df2.dtypes)

DF1 columns: Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')
DF2 columns: Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

DF1 types:
 InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

DF2 types:
 Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object


## 3. Transform - integracja

Ujednolicamy:
- nazwy kolumn
- typy danych
- format dat

### Który schemat przyjąć jako docelowy?
Jako schemat docelowy przyjęto strukturę pierwszego datasetu (Online_Retail.csv).

### Czy wszystkie kolumny są potrzebne?
Nie. W tabeli faktów używamy tylko kluczowych kolumn.

In [3]:
# ujednolicenie nazw kolumn
df2 = df2.rename(columns={
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID"
})

In [4]:
def clean_data(df):
    # konwersja na liczby
    df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
    df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
    df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")

    # usunięcie braków
    df = df.dropna(subset=["CustomerID", "Quantity", "UnitPrice"])

    # filtrowanie błędów
    df = df[df["Quantity"] > 0]
    df = df[df["UnitPrice"] >= 0]

    # duplikaty
    df = df.drop_duplicates()

    # daty
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], dayfirst=True, format="mixed")

    # typy
    df["CustomerID"] = df["CustomerID"].astype(int)

    # miara
    df["Revenue"] = (df["Quantity"] * df["UnitPrice"]).round(2)

    return df

In [5]:
df1 = clean_data(df1)
df2 = clean_data(df2)

## 4. Data Quality

Sprawdzamy jakość danych.

### Jak rozpoznać duplikat?
Duplikat to rekord o identycznych wartościach we wszystkich kolumnach.

### Co zrobić z konfliktem danych?
Można:
- usunąć duplikaty
- wybrać bardziej wiarygodne źródło
- zastosować reguły biznesowe

### Które źródło jest bardziej wiarygodne?
Źródło z mniejszą liczbą błędów i brakujących wartości.

In [6]:
# sprawdzanie duplikatów między zbiorami
df_temp = pd.concat([df1, df2])

duplicates = df_temp[df_temp.duplicated()]

print("Liczba duplikatów:", len(duplicates))

Liczba duplikatów: 0


## 5. Merge - łączenie danych

Łączymy dane po wcześniejszym czyszczeniu.

### Czy użyć concat czy merge?
Użyto concat, ponieważ dane mają tę samą strukturę i są łączone pionowo.

### Czy zachować wszystkie rekordy?
Nie. Po połączeniu należy usunąć duplikaty.

In [7]:
df_all = pd.concat([df1, df2], ignore_index=True)

# usunięcie duplikatów
df_all = df_all.drop_duplicates()

df_all.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-01-12 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-01-12 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-01-12 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-01-12 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-01-12 08:26:00,3.39,17850,United Kingdom,20.34


## 6. Load - zapis danych

Zapisujemy wynik jako jedną tabelę faktów.

In [8]:
df_all.to_csv("fact_sales_integrated.csv", index=False)

## Podsumowanie

- wczytano dane z CSV i Excel
- ujednolicono strukturę
- oczyszczono dane
- usunięto duplikaty
- połączono dane w jedną tabelę faktów

Proces integracji danych jest kluczowy w ETL.